# CerberusVision Phase 5.1 — Qwen-2.5-7B-Instruct From-Scratch QLoRA

Bu defter, **sifirdan (from-scratch)** QLoRA fine-tuning yapar.
Phase 5'in yerine gececek temiz bir adapter uretir.

**Phase 5.1 Stratejisi:**
- Sifirdan QLoRA — Phase 5 adapter YUKLENMEZ
- Tum veri: %100 Phase 5 + %100 Turkce BL + %100 Reefer + %100 Coklu Konteyner
- Global aile bazli split — sifir sizinti garantisi
- Erken durdurma: patience=2, eval_steps=10

**Veri (2026-07-25 - Guclendirilmis):**
- Train: 1156 ornek (828 Phase 5 + 100 TR BL + 180 Coklu Konteyner + 48 Reefer)
- Validation: 215 ornek, 2 aile (global split, sifir cakisma)
- Toplam: 1371 kayit, 122 aile
- Veri dagilimi: %72 Phase 5 / %16 Coklu Konteyner / %9 TR BL / %4 Reefer

**Drive Dizini Yapisi:**
```
MyDrive/CerberusVision_Phase5_1_Colab/
├── data/
│   ├── train.jsonl          (~2.9 MB)
│   ├── validation.jsonl     (~650 KB)
│   └── manifest.json
├── checkpoints/          ← egitim sirasinda Drive'a yazilir
└── adapter_best/         ← egitim sonunda buraya kaydedilir
```

In [1]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 49.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
from pathlib import Path
import shutil
import json

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/CerberusVision_Phase5_1_Colab")
DATA_DIR = DRIVE_DIR / "data"
CHECKPOINT_DIR = DRIVE_DIR / "checkpoints"
FINAL_ADAPTER_DIR = DRIVE_DIR / "adapter_best"

LOCAL_DATA_DIR = Path("/content/phase5_1_data")

for d in [DATA_DIR, CHECKPOINT_DIR, FINAL_ADAPTER_DIR, LOCAL_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Drive mount tamam. Dizinler hazir.")
print(f"  Veri (Drive):        {DATA_DIR}")
print(f"  Checkpoint (Drive):   {CHECKPOINT_DIR}")
print(f"  Final adapter:        {FINAL_ADAPTER_DIR}")
print(f"  Veri (lokal):         {LOCAL_DATA_DIR}")

Mounted at /content/drive
Drive mount tamam. Dizinler hazir.
  Veri (Drive):        /content/drive/MyDrive/CerberusVision_Phase5_1_Colab/data
  Checkpoint (Drive):   /content/drive/MyDrive/CerberusVision_Phase5_1_Colab/checkpoints
  Final adapter:        /content/drive/MyDrive/CerberusVision_Phase5_1_Colab/adapter_best
  Veri (lokal):         /content/phase5_1_data


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

model_id = "Qwen/Qwen2.5-7B-Instruct"
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"bf16 supported: {use_bf16}, compute_dtype: {compute_dtype}")

# ============================================================
# Step 1: Load base model (same as Phase 5)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|endoftext|>'})

print(f"PAD Token ID: {tokenizer.pad_token_id}")
print(f"EOS Token ID: {tokenizer.eos_token_id}")
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD == EOS!"
assert tokenizer.eos_token == "<|im_end|>", f"EOS token mismatch: {tokenizer.eos_token}"

# ============================================================
# Step 2: Apply fresh LoRA (from-scratch, NOT from Phase 5)
# ============================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("\nFresh LoRA adapter applied — from-scratch training'e hazir.")

bf16 supported: True, compute_dtype: torch.bfloat16


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

PAD Token ID: 151643
EOS Token ID: 151645
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273

Fresh LoRA adapter applied — from-scratch training'e hazir.


In [4]:
print("Phase 5.1 verileri Drive'dan lokal /content calisma alanina kopyalaniyor...")

expected_files = {
    "train.jsonl": "Phase 5.1 egitim verisi (1156 ornek)",
    "validation.jsonl": "Phase 5.1 dogrulama verisi (215 ornek)",
}

missing = []
for filename, desc in expected_files.items():
    source = DATA_DIR / filename
    destination = LOCAL_DATA_DIR / filename
    if not source.exists():
        missing.append((filename, desc))
        continue
    shutil.copy2(source, destination)
    with open(destination, 'r') as f:
        line_count = sum(1 for _ in f)
    print(f"  Kopyalandi: {filename} ({line_count} satir)")

if missing:
    print("\n*** EKSIK DOSYALAR — Google Drive'a yuklemen gerekiyor ***\n")
    print(f"Hedef klasor: {DATA_DIR}")
    print()
    for filename, desc in missing:
        print(f"  • {filename}  ({desc})")
    print()
    raise FileNotFoundError(f"{len(missing)} dosya Drive'da bulunamadi.")

print("\nVeriler Drive'dan lokal /content calisma alanina tasindi.")

Phase 5.1 verileri Drive'dan lokal /content calisma alanina kopyalaniyor...
  Kopyalandi: train.jsonl (1156 satir)
  Kopyalandi: validation.jsonl (215 satir)

Veriler Drive'dan lokal /content calisma alanina tasindi.


In [5]:
# ============================================================
# Dataset hazirlama
# ============================================================
train_dataset = load_dataset(
    "json",
    data_files=str(LOCAL_DATA_DIR / "train.jsonl"),
    split="train"
)
eval_dataset = load_dataset(
    "json",
    data_files=str(LOCAL_DATA_DIR / "validation.jsonl"),
    split="train"
)

def format_dataset(example):
    return {
        "prompt": [
            {"role": "system", "content": "Extract shipping instruction data from OCR text as JSON."},
            {"role": "user", "content": str(example['input'])}
        ],
        "completion": [
            {"role": "assistant", "content": str(example['output'])}
        ]
    }

train_dataset = train_dataset.map(format_dataset, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_dataset, remove_columns=eval_dataset.column_names)

print(f"Train size: {len(train_dataset)}")
print(f"Eval size:  {len(eval_dataset)}")
print(f"Train/Eval ratio: {len(eval_dataset)/(len(train_dataset)+len(eval_dataset))*100:.1f}%")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1156 [00:00<?, ? examples/s]

Map:   0%|          | 0/215 [00:00<?, ? examples/s]

Train size: 1156
Eval size:  215
Train/Eval ratio: 15.7%


In [6]:
# ============================================================
# Phase 5.1 Training Config (from-scratch)
# ============================================================
# Kritik degisiklikler vs Phase 5:
#   - Tum veriyle from-scratch (Phase 5 adapter yok)
#   - eval_steps: 20 → 10 (daha sik degerlendirme)
#   - early_stopping_patience: 3 → 2 (overfitting'i erken yakala)
#   - seed: 42 → 3407

RESUME_TRAINING = False  # Runtime koparsa True yap

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),

    # Batch (Phase 5 ile ayni)
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Etkin batch = 16

    # LR (Phase 5 ile ayni — from-scratch oldugu icin)
    learning_rate=5e-5,
    num_train_epochs=3,
    warmup_steps=50,
    lr_scheduler_type="cosine",

    # Daha sik degerlendirme + erken durdurma
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Erken durdurma — Phase 5'teki gecikmeyi onler

    # Logging
    logging_steps=5,

    # Donanim
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_32bit",

    # Seed (Phase 5.1)
    seed=3407,
    data_seed=3407,

    # SFT ozel
    report_to="none",
    max_length=2048,
    completion_only_loss=True,
    eos_token="<|im_end|>",
    packing=False,
)

print("Phase 5.1 Training Config (from-scratch):")
print(f"  Learning Rate:    {training_args.learning_rate}")
print(f"  Epochs:           {training_args.num_train_epochs}")
print(f"  Batch Size:       {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = 16")
print(f"  Eval Steps:       {training_args.eval_steps}")
print("  Early Stopping:   Enabled (via EarlyStoppingCallback, patience=2, threshold=0.001)\n")
print(f"  Max Seq Length:   {training_args.max_length}")
print(f"  Seed:             {training_args.seed}")
print(f"  Resume Training:  {RESUME_TRAINING}")
print()

# ============================================================
# Trainer
# ============================================================
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.001)],
)

# ============================================================
# Train
# ============================================================
print("Egitim basliyor...")
print(f"  Train: {len(train_dataset)} ornek")
print(f"  Eval:  {len(eval_dataset)} ornek")
print(f"  Mode:  FROM-SCRATCH (taze LoRA, Phase 5 adapter yok)")
print()

train_result = trainer.train(
    resume_from_checkpoint=True if RESUME_TRAINING else None
)

# ============================================================
# Save adapter
# ============================================================
trainer.model.save_pretrained(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

# Training metrics
metrics = {
    **train_result.metrics,
    "best_eval_loss": trainer.state.best_metric,
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "phase": "5.1",
    "training_type": "from_scratch",
}

metrics_path = FINAL_ADAPTER_DIR / "training_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"\nPhase 5.1 adapter Drive'a kaydedildi: {FINAL_ADAPTER_DIR}")
print(f"  Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"  Best eval loss:  {trainer.state.best_metric}")
print(f"  Metrikler:       {metrics_path}")

Phase 5.1 Training Config (from-scratch):
  Learning Rate:    5e-05
  Epochs:           3
  Batch Size:       2 x 8 = 16
  Eval Steps:       10
  Early Stopping:   Enabled (via EarlyStoppingCallback, patience=2, threshold=0.001)

  Max Seq Length:   2048
  Seed:             3407
  Resume Training:  False



Tokenizing train dataset:   0%|          | 0/1156 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1156 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1156 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1156 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/215 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Egitim basliyor...
  Train: 1156 ornek
  Eval:  215 ornek
  Mode:  FROM-SCRATCH (taze LoRA, Phase 5 adapter yok)



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,0.693048,0.588084,0.196720,192783.000000,0.890989
20,0.546329,0.423208,0.238944,392135.000000,0.907461
30,0.314851,0.210680,0.261227,588062.000000,0.942451
40,0.134656,0.099828,0.094064,779018.000000,0.977503
50,0.054605,0.105389,0.043114,962711.000000,0.981699
60,0.033706,0.119801,0.033111,1154573.000000,0.982657



Phase 5.1 adapter Drive'a kaydedildi: /content/drive/MyDrive/CerberusVision_Phase5_1_Colab/adapter_best
  Best checkpoint: /content/drive/MyDrive/CerberusVision_Phase5_1_Colab/checkpoints/checkpoint-40
  Best eval loss:  0.09982821345329285
  Metrikler:       /content/drive/MyDrive/CerberusVision_Phase5_1_Colab/adapter_best/training_metrics.json


### Runtime Koparsa Devam Etmek Icin

Eger Colab runtime'i koparsa:

1. **Cell 1'den Cell 6'ya kadar** sirasiyla yeniden calistir
2. **Cell 7'de `RESUME_TRAINING = False` yerine `RESUME_TRAINING = True` yap**
3. Cell 7'yi calistir

```python
RESUME_TRAINING = True   # ← bununla degistir
```

### Egitim Sonrasi

1. `adapter_best/` klasorunu bilgisayara indir
2. Projede `models/Qwen-2.5-7B-Instruct-Phase5_1-LoRA/` olarak kaydet
3. Benchmark'i calistir:
```bash
.venv/bin/python scripts/benchmark_accuracy.py tests/fixtures/qwen_benchmark \
  --output benchmark_results_phase5_1.json \
  --html benchmark_report_phase5_1.html
```